In [180]:
from inequality.polarization import S

In [181]:
import libpysal
import pandas as pd
import numpy as np
from libpysal.weights import lat2W
from libpysal.graph import Graph

In [182]:
y = np.arange(1600)

In [183]:
df = pd.DataFrame({'y': y}, index=y)

In [184]:
g = Graph.from_W(lat2W(40, 40))

In [185]:
%%timeit
res = S(df, g, 'y', n_jobs=4)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 999/999 [00:00<00:00, 2497.69it/s]


sim.mean()=np.float64(0.9192246063334923)
sim.std()=np.float64(0.005869381067509393)


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 999/999 [00:00<00:00, 2507.35it/s]


sim.mean()=np.float64(0.9192515419048586)
sim.std()=np.float64(0.0057810914702703204)


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 999/999 [00:00<00:00, 2562.17it/s]


...

sim.mean()=np.float64(0.9194256835057836)
sim.std()=np.float64(0.005748407039336092)


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 999/999 [00:00<00:00, 2617.28it/s]


sim.mean()=np.float64(0.919414408150328)
sim.std()=np.float64(0.006019827018236639)


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 999/999 [00:00<00:00, 2549.64it/s]


sim.mean()=np.float64(0.9190986981975718)
sim.std()=np.float64(0.00601317315464993)
740 ms ± 13.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [174]:
res[0]

1.0

In [175]:
res[1]

np.float64(0.001)

In [176]:
res[2].groupby(by='i_labels').count()

,a_labels,g_labels
i_labels,,
0,800,800
1,800,800


In [177]:
res[2].groupby(by='a_labels').count()

,i_labels,g_labels
a_labels,,
0,800,800
1,800,800


In [178]:
res[2].groupby(by='g_labels').count()

,i_labels,a_labels
g_labels,,
0,1600,1600


In [126]:
type(g.adjacency)

pandas.core.series.Series

In [127]:
g.adjacency.index

MultiIndex([( 0,  1),
            ( 0,  4),
            ( 1,  0),
            ( 1,  2),
            ( 1,  5),
            ( 2,  1),
            ( 2,  3),
            ( 2,  6),
            ( 3,  2),
            ( 3,  7),
            ( 4,  0),
            ( 4,  5),
            ( 4,  8),
            ( 5,  1),
            ( 5,  4),
            ( 5,  6),
            ( 5,  9),
            ( 6,  2),
            ( 6,  5),
            ( 6,  7),
            ( 6, 10),
            ( 7,  3),
            ( 7,  6),
            ( 7, 11),
            ( 8,  4),
            ( 8,  9),
            ( 8, 12),
            ( 9,  5),
            ( 9,  8),
            ( 9, 10),
            ( 9, 13),
            (10,  6),
            (10,  9),
            (10, 11),
            (10, 14),
            (11,  7),
            (11, 10),
            (11, 15),
            (12,  8),
            (12, 13),
            (13,  9),
            (13, 12),
            (13, 14),
            (14, 10),
            (14, 13),
          

In [128]:
y

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15])

In [144]:
import pandas as pd
import networkx as nx
import numpy as np
from tqdm import trange
from joblib import Parallel, delayed

def S(df, g, column, k=2, bins=None, permutations=999,
      seed=None, keep_sim=False, n_jobs=1, verbose=True):
    """Compute a spatial polarization index for a variable.

    This function measures the degree of spatial polarization by
    comparing the alignment between categorical groupings of a
    variable and the connectivity structure of a spatial graph.

    A higher index value indicates stronger spatial polarization of the values.

    The observed index is compared against a null distribution
    generated via Monte Carlo permutation, producing an empirical
    p-value to assess statistical significance.

    Parameters
    ----------
    df : pandas.DataFrame
         The dataframe containing spatial observations. Its index must
         align with the nodes in the PySAL spatial graph object `g`.

    g : libpysal.graph.Graph
        A PySAL spatial Graph object representing spatial
        connectivity. Internally converted to a NetworkX graph to
        evaluate component structure.

    column : str
        The name of the column in `df` to analyze for spatial polarization.

    bins : list of float, optional
        Cut points for binning the variable into discrete categories.
        If `None` (default), the variable is split at the median into
        two groups.

    permutations : int, default 999
        Number of permutations used to generate the null distribution
        for inference.

    seed : int or None, optional
        Random seed for reproducibility of the permutation test.

    keep_sim : bool, default False
        Whether to return the full array of simulated polarization scores.

    n_jobs : int, default 1
        The number of jobs to run in parallel. `1` means no parallelization.
        `-1` means using all available CPU cores.

    verbose : bool, default True
        If True, print the mean and standard deviation of the simulated
        polarization scores. Also controls the `tqdm` progress bar.

    Returns
    -------
    s : float
        The observed spatial polarization index, bounded in [0, 1].
        Higher values indicate stronger spatial separation of the
        attribute groups.

    p_value : float
        Monte Carlo p-value indicating how extreme the observed index is
        under random label assignment.

    sim : numpy.ndarray, optional
        Array of simulated polarization indices from the permutation
        distribution. Returned only if `keep_sim` is True.

    Notes
    -----
    - The polarization index is based on connected components in a subgraph
      formed from edges linking observations in the same category.
    - The observed index reflects the relative reduction in fragmentation
      compared to a randomized assignment.

    Example
    -------
    >>> import libpysal
    >>> import pandas as pd
    >>> import numpy as np
    >>> from libpysal.weights import lat2W
    >>> from libpysal.graph import Graph

    >>> y = np.arange(1600)
    >>> df = pd.DataFrame({'y':y}, index=y)
    >>> g = Graph.from_W(lat2W(40, 40))
    >>> S(df, g, 'y', permutations=99, seed=1)
    (1.0, 0.01)

    """
    n = df.shape[0]
    if bins is not None:
        Ca = len(bins) - 1
        labels = range(Ca)
        clique = pd.cut(df[column], bins=bins, labels=labels)
    else:
        if not isinstance(k, int) or k < 1 or k > n:
            raise ValueError("'k' must be a positive integer less than n.")
        clique = pd.qcut(df[column], q=k, labels=False, duplicates='drop')
        Ca = k
        labels = range(Ca)

    Gg = g.to_networkx()
    Cg = nx.number_connected_components(Gg)
    k = max(Cg, Ca)
    focal = g.adjacency.index.get_level_values(0)
    neighbor = g.adjacency.index.get_level_values(1)

    def _calc(clique_labels, n, k):
        left = clique_labels.loc[focal].values
        right = clique_labels.loc[neighbor].values
        edges = g.adjacency[left == right]
        i = edges.index.get_level_values(0)
        j = edges.index.get_level_values(1)
        edges = zip(i, j)
        visited = np.zeros(n, int)
        labels = np.zeros_like(visited)
        c = 0  # number of components in intersection graph
        for edge in edges:
            i, j = edge
            print(f'\n\n{i=}, {j=}, {c=}, {labels=}, {visited=}')
            if visited[i] == visited[j]:
                if visited[i] == 0:
                    # new component
                    c += 1
                    labels[i] = c
                    labels[j] = c
                    visited[i] = 1
                    visited[j] = 1
                else:
                    if labels[i] != labels[j]:
                        # bridge edge, merge components
                        if labels[i] > labels[j]:
                            labels[labels==labels[i]] = labels[j]
                        else:
                            labels[labels==labels[j]] = labels[i]
                            c -= 1
            elif visited[i] == 0:
                # new node, grow component
                labels[i] = labels[j]
                visited[i] = 1
            else:
                # new node, grow component
                labels[j] = labels[i]
                visited[j] = 1
            print(f'{i=}, {j=}, {c=}, {labels=}, {visited=}')
        print(c)
        print(labels)
        return 1 - (c - k) / (n - k)

    s = _calc(clique, n, k)

    sim = np.zeros(permutations)
    rng = np.random.default_rng(seed)

    def permute_and_calc(v, index, n, k, seed_i):
        rng_i = np.random.default_rng(seed_i)
        shuffled = pd.Series(rng_i.permutation(v), index=index)
        return _calc(shuffled, n, k)

    v = np.array(clique)
    seeds = rng.integers(low=0, high=1e9, size=permutations)
    sim = Parallel(n_jobs=n_jobs)(
        delayed(permute_and_calc)(v, range(n), n, k, seeds[current_seed])
        for current_seed in trange(permutations)
        )
    sim = np.array(sim)
    if verbose:
        print(f'{sim.mean()=}')
        print(f'{sim.std()=}')
    p_value = ((sim >= s).sum()+1) / (permutations+1)
    if keep_sim:
        return s, p_value, sim
    else:
        return s, p_value


In [145]:
res = S(df, g, 'y', permutations=0)



i=0, j=1, c=0, labels=array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), visited=array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
i=0, j=1, c=1, labels=array([1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), visited=array([1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])


i=0, j=4, c=1, labels=array([1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), visited=array([1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
i=0, j=4, c=1, labels=array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), visited=array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])


i=1, j=0, c=1, labels=array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), visited=array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
i=1, j=0, c=1, labels=array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), visited=array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])


i=1, j=2, c=1, labels=array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), visited=array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0

0it [00:00, ?it/s]

sim.mean()=np.float64(nan)
sim.std()=np.float64(nan)



/tmp/ipykernel_574440/2910495847.py:173: RuntimeWarning: Mean of empty slice.
  print(f'{sim.mean()=}')
/home/serge/miniforge3/envs/dev/lib/python3.12/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/serge/miniforge3/envs/dev/lib/python3.12/site-packages/numpy/_core/_methods.py:223: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/serge/miniforge3/envs/dev/lib/python3.12/site-packages/numpy/_core/_methods.py:181: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/serge/miniforge3/envs/dev/lib/python3.12/site-packages/numpy/_core/_methods.py:215: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [146]:
res[0]

1.0

In [117]:
edges = res[-1]

In [118]:
for edge in edges:
    i,j = edge
    print(i,j)

In [101]:
def components_from_edges(edges, n):
    
    visited = np.zeros(n, int)
    labels = np.zeros_like(visited)
    c = 0
    for edge in edges:
        i, j = edge
        if visited[i] == visited[j]:
            if visited[i] == 0:
                # new component
                c += 1
                labels[i] = c
                labels[j] = c
                visited[i] = 1
                visited[j] = 1
            else:
                if labels[i] != labels[j]:
                    # bridge edge, merge components
                    if labels[i] > labels[j]:
                        labels[labels==labels[i]] = labels[j]
                    else:
                        labels[labels==labels[j]] = labels[i]
                    c -= 1
        elif visited[i] == 0:
            # new node, grow component
            labels[i] = labels[j]
            visited[i] = 1
        else:
            # new node, grow component
            labels[j] = labels[i]
            visited[j] = 1
    return (labels, c)
            

In [102]:
components_from_edges(edges, 16)

(array([1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2]), 2)

In [103]:
res = S(df, g, 'y', permutations=99, seed=1)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 99/99 [00:00<00:00, 1639.33it/s]

sim.mean()=np.float64(0.7323232323232322)
sim.std()=np.float64(0.08269292226201679)


In [105]:
S??

Signature:
S(
    df,
    g,
    column,
    k=2,
    bins=None,
    permutations=999,
    seed=None,
    keep_sim=False,
    n_jobs=1,
    verbose=True,
)
Docstring:
Compute a spatial polarization index for a variable.

This function measures the degree of spatial polarization by
comparing the alignment between categorical groupings of a
variable and the connectivity structure of a spatial graph.

A higher index value indicates stronger spatial polarization of the values.

The observed index is compared against a null distribution
generated via Monte Carlo permutation, producing an empirical
p-value to assess statistical significance.

Parameters
----------
df : pandas.DataFrame
     The dataframe containing spatial observations. Its index must
     align with the nodes in the PySAL spatial graph object `g`.

g : libpysal.graph.Graph
    A PySAL spatial Graph object representing spatial
    connectivity. Internally converted to a NetworkX graph to
    evaluate component structure.

co

In [99]:
res

(0.5714285714285714, np.float64(1.0))

In [28]:
res = S(df, g, 'y', permutations=99, seed=1, n_jobs=4)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 99/99 [00:01<00:00, 89.46it/s]


sim.mean()=np.float64(0.6327796108772329)
sim.std()=np.float64(0.007640646047079661)


In [ ]:
%%timeit
res = S(df, g, 'y', permutations=999, seed=1)

In [ ]:
%%timeit
res = S(df, g, 'y', permutations=999, seed=1, n_jobs=1)

In [ ]:
%%timeit
res = S(df, g, 'y', permutations=999, seed=1, n_jobs=40)

In [ ]:
%%timeit
res = S(df, g, 'y', permutations=999, seed=1, n_jobs=8)